In [1]:
# 04_rcfs.ipynb
# Regulatory-Constrained Fairness Score for Credit Scoring
# Notebook 04 - The Regulatory-Constrained Fairness Score (RCFS).
#
# RCFS fuses predictive adequacy with a harmonic combination of regulatory-
# requirement satisfaction scores. Each component lies in [0, 1] (higher = better):
#   c_perf : discriminative adequacy = max(0, (AUC - 0.5) / 0.5)
#   c_fair : fairness on the FAVOURABLE outcome (non-default), worst-case across
#            ALL sensitive attributes of the dataset:
#              per attribute a, compare the pooled unprivileged group against the
#              privileged reference group on the favourable-decision rate;
#              c_fair(a) = 0.5*min(1, DI_a) + 0.5*(1 - |SPD_a|);
#              c_fair    = min_a c_fair(a)   (no attribute may be sacrificed)
#   c_expl : explainability = min(1, k_star / n_eff), where n_eff = exp(entropy of
#            the mean |SHAP| distribution). k_star is the target reason-code budget.
# Compliance is the weighted HARMONIC mean of {c_fair, c_expl}; RCFS trades
# performance against compliance through gamma:
#   Compliance = (sum w_k) / (sum w_k / c_k);  RCFS = (1-gamma)*c_perf + gamma*Compliance
#
# Reviewer-driven changes made explicit here:
#   - EXPL_KSTAR (k_star) is stated as a parameter and reported in the output.
#   - Fairness is measured on the favourable (non-default) decision; privileged
#     reference groups come from the metadata; the two-attribute datasets combine
#     into a single c_fair by the worst-case rule above.
#   - All decision thresholds (baseline prevalence threshold and reject-option
#     group thresholds) are tuned on the VALIDATION split and frozen for TEST.
#   - A constrained-optimisation technique (exponentiated gradient, demographic
#     parity) is added alongside reweighing and reject-option.
#   - Per-technique disparate impact, |SPD|, and effective feature count are printed.
#   - The compliance aggregator is compared against geometric and arithmetic means
#     to show whether the harmonic choice changes the technique ranking.
# The canonical classifier and XGB_PARAMS are identical to notebook 03, so the
# baseline AUC here equals the XGBoost row of Table 2. No paths are printed.

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
EXPL_KSTAR = 10          # target number of effective reason-code features (stated)
W_FAIR, W_EXPL = 1.0, 1.0
GAMMA_DEFAULT = 0.5
GAMMA_SWEEP = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
XGB_PARAMS = dict(n_estimators=400, max_depth=5, learning_rate=0.05,
                  subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                  random_state=RANDOM_STATE, n_jobs=-1)
TECHNIQUES = ["baseline", "reweighing", "reject_option", "exp_gradient"]

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)
DATASETS = [("German Credit", "german"), ("Taiwan CC Default", "taiwan"),
            ("Credit Card Approval", "credit_card_approval")]


def load_split(key, part):
    return pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_{part}.parquet"))


def get_Xy(df, key):
    meta = META[key]
    drop = [meta["target_col"]] + meta["sensitive_cols"]
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    y = df[meta["target_col"]].astype(int)
    return X, y


def priv_mask_for(series, privileged):
    # Boolean mask of the privileged reference group for one sensitive attribute.
    if isinstance(privileged, (list, tuple)):
        return series.isin(list(privileged)).values
    return (series == privileged).values


def di_spd_favourable(y_pred, priv_mask, unpriv_mask, favourable_label):
    # Disparate impact and statistical parity difference on the FAVOURABLE outcome.
    fav = (y_pred == favourable_label)
    r_p = fav[priv_mask].mean() if priv_mask.sum() else np.nan
    r_u = fav[unpriv_mask].mean() if unpriv_mask.sum() else np.nan
    if not (np.isfinite(r_p) and np.isfinite(r_u)) or r_p == 0:
        return np.nan, np.nan
    return r_u / r_p, r_u - r_p


def c_fair_worstcase(test_df, key, y_pred):
    # Worst-case fairness satisfaction across all sensitive attributes; also returns
    # the attribute-level DI and |SPD| for the binding (worst) attribute.
    meta = META[key]
    fav = meta["favourable_label"]
    best = None
    worst_di, worst_spd, worst_attr = np.nan, np.nan, None
    for a in meta["sensitive_cols"]:
        s = test_df[a]
        pm = priv_mask_for(s, meta["privileged"][a])
        um = (~pm) & s.notna().values
        di, spd = di_spd_favourable(np.asarray(y_pred), pm, um, fav)
        if not np.isfinite(di):
            continue
        score = 0.5 * min(1.0, di) + 0.5 * (1.0 - abs(spd))
        if best is None or score < best:
            best, worst_di, worst_spd, worst_attr = score, di, abs(spd), a
    return best, worst_di, worst_spd, worst_attr


def shap_neff(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    try:
        sv = shap.TreeExplainer(model).shap_values(X)
    except Exception:
        sv = shap.Explainer(model, X)(X).values
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0:
        return np.nan
    p = mean_abs / total
    p = p[p > 0]
    if len(p) <= 1:
        return 1.0
    return float(np.exp(-np.sum(p * np.log(p))))


def c_expl_from_neff(n_eff, k_star=EXPL_KSTAR):
    return np.nan if not np.isfinite(n_eff) else float(min(1.0, k_star / n_eff))


def val_prevalence_threshold(proba_val, y_train_rate):
    return np.quantile(proba_val, 1.0 - y_train_rate)


def reject_option_thresholds(proba_val, s_val, rate):
    # Group thresholds tuned on VALIDATION so each group's positive rate matches
    # the overall prevalence; frozen and applied to the TEST split.
    thr = {}
    for grp in pd.unique(s_val):
        p = proba_val[(s_val.values == grp)]
        thr[grp] = np.quantile(p, 1.0 - rate) if len(p) else 0.5
    return thr


def group_reweighing_weights(y, s):
    w = np.ones(len(y), dtype=float); n = len(y)
    for yy in np.unique(y):
        for ss in pd.unique(s):
            mask = (y == yy) & (s.values == ss)
            if mask.sum() == 0:
                continue
            w[mask] = ((y == yy).mean() * (s.values == ss).mean()) / (mask.sum() / n)
    return w


def spw_of(y):
    n_pos = int(y.sum()); n_neg = int((y == 0).sum())
    return (n_neg / n_pos) if n_pos > 0 else 1.0


def eg_proba(eg, X):
    # Weighted-average positive-class probability over the ExponentiatedGradient
    # ensemble, to obtain continuous scores for AUC.
    w = np.asarray(eg.weights_, dtype=float)
    keep = w > 0
    w = w[keep]; w = w / w.sum()
    preds = eg.predictors_
    probs = np.zeros(len(X))
    j = 0
    for i, pk in enumerate(preds):
        if not keep[i]:
            continue
        probs += w[j] * pk.predict_proba(X)[:, 1]
        j += 1
    return probs


def run_technique(tech, X_tr, y_tr, X_val, X_te, s_tr, s_val, s_te, prim_sens):
    # Returns (model_for_shap, proba_test, y_pred_test). All thresholds use validation.
    base = XGBClassifier(scale_pos_weight=spw_of(y_tr), **XGB_PARAMS)
    rate = float(y_tr.mean())
    if tech == "baseline":
        base.fit(X_tr, y_tr)
        proba_val = base.predict_proba(X_val)[:, 1]
        proba_te = base.predict_proba(X_te)[:, 1]
        thr = val_prevalence_threshold(proba_val, rate)
        return base, proba_te, (proba_te >= thr).astype(int)
    if tech == "reweighing":
        w = group_reweighing_weights(y_tr.values, s_tr)
        base.fit(X_tr, y_tr, sample_weight=w)
        proba_val = base.predict_proba(X_val)[:, 1]
        proba_te = base.predict_proba(X_te)[:, 1]
        thr = val_prevalence_threshold(proba_val, rate)
        return base, proba_te, (proba_te >= thr).astype(int)
    if tech == "reject_option":
        base.fit(X_tr, y_tr)
        proba_val = base.predict_proba(X_val)[:, 1]
        proba_te = base.predict_proba(X_te)[:, 1]
        tmap = reject_option_thresholds(proba_val, s_val, rate)
        yhat = np.array([int(p >= tmap.get(g, 0.5)) for p, g in zip(proba_te, s_te.values)])
        return base, proba_te, yhat
    if tech == "exp_gradient":
        # In-processing constrained optimisation: demographic parity on the primary
        # sensitive attribute. Fairness constraint is enforced during fitting; the
        # decision rule is the ensemble's 0.5 threshold (no post-hoc tuning).
        est = XGBClassifier(scale_pos_weight=spw_of(y_tr), **XGB_PARAMS)
        eg = ExponentiatedGradient(est, constraints=DemographicParity(), max_iter=25)
        eg.fit(X_tr, y_tr, sensitive_features=s_tr.values)
        proba_te = eg_proba(eg, X_te)
        yhat = np.asarray(eg.predict(X_te)).astype(int)
        # Use a representative ensemble member for SHAP (explainability proxy).
        w = np.asarray(eg.weights_, dtype=float)
        rep = eg.predictors_[int(np.argmax(w))]
        return rep, proba_te, yhat
    raise ValueError(tech)


def compliance(cs, ws, kind="harmonic"):
    pairs = [(c, w) for c, w in zip(cs, ws) if np.isfinite(c) and c > 0]
    if not pairs:
        return np.nan
    if kind == "harmonic":
        return sum(w for _, w in pairs) / sum(w / c for c, w in pairs)
    if kind == "geometric":
        wsum = sum(w for _, w in pairs)
        return float(np.exp(sum(w * np.log(c) for c, w in pairs) / wsum))
    if kind == "arithmetic":
        wsum = sum(w for _, w in pairs)
        return sum(w * c for c, w in pairs) / wsum
    raise ValueError(kind)


def rcfs(c_perf, comp, gamma):
    return np.nan if not np.isfinite(comp) else (1 - gamma) * c_perf + gamma * comp


rows, agg_rows, detail = [], [], {}
for display_name, key in DATASETS:
    meta = META[key]
    prim = meta["sensitive_cols"][0]
    tr, val, te = load_split(key, "train"), load_split(key, "val"), load_split(key, "test")
    X_tr, y_tr = get_Xy(tr, key); X_val, _ = get_Xy(val, key); X_te, y_te = get_Xy(te, key)
    X_val = X_val[X_tr.columns]; X_te = X_te[X_tr.columns]
    s_tr, s_val, s_te = tr[prim], val[prim], te[prim]
    for tech in TECHNIQUES:
        model, proba, y_pred = run_technique(
            tech, X_tr, y_tr, X_val, X_te, s_tr, s_val, s_te, prim)
        auc = roc_auc_score(y_te.values, proba)
        c_perf = max(0.0, (auc - 0.5) / 0.5)
        c_fair, di, spd, binding = c_fair_worstcase(te, key, y_pred)
        n_eff = shap_neff(model, X_te)
        c_expl = c_expl_from_neff(n_eff)
        comp_h = compliance([c_fair, c_expl], [W_FAIR, W_EXPL], "harmonic")
        score = rcfs(c_perf, comp_h, GAMMA_DEFAULT)
        rows.append({"dataset": display_name, "technique": tech, "AUC": auc,
                     "c_perf": c_perf, "c_fair": c_fair, "c_expl": c_expl,
                     "DI": di, "abs_SPD": spd, "n_eff": n_eff,
                     "binding_attr": binding, "Compliance": comp_h, "RCFS": score})
        detail[(display_name, tech)] = {"c_perf": c_perf, "compliance": comp_h}
        # Ranking under the three aggregators (gamma fixed at default).
        for kind in ("harmonic", "geometric", "arithmetic"):
            cc = compliance([c_fair, c_expl], [W_FAIR, W_EXPL], kind)
            agg_rows.append({"dataset": display_name, "technique": tech,
                             "aggregator": kind, "RCFS": rcfs(c_perf, cc, GAMMA_DEFAULT)})
        print(f"[{display_name:>22}] {tech:<13} AUC={auc:.3f} c_fair={c_fair:.3f} "
              f"c_expl={c_expl:.3f} DI={di:.3f} |SPD|={spd:.3f} n_eff={n_eff:.1f} "
              f"RCFS={score:.3f}")

rcfs_df = pd.DataFrame(rows).round(4)
print(rcfs_df.to_string(index=False))
rcfs_df.to_csv(os.path.join(TAB_DIR, "04_rcfs_by_technique.csv"), index=False)

best = rcfs_df.loc[rcfs_df.groupby("dataset")["RCFS"].idxmax()][
    ["dataset", "technique", "RCFS", "c_perf", "Compliance"]]
print("Best technique by RCFS (point estimate):")
print(best.to_string(index=False))
best.to_csv(os.path.join(TAB_DIR, "04_rcfs_best_technique.csv"), index=False)

# Gamma sweep for every (dataset, technique).
sweep_rows = []
for (display_name, tech), comp in detail.items():
    for g in GAMMA_SWEEP:
        sweep_rows.append({"dataset": display_name, "technique": tech, "gamma": g,
                           "RCFS": rcfs(comp["c_perf"], comp["compliance"], g)})
sweep_df = pd.DataFrame(sweep_rows).round(4)
sweep_df.to_csv(os.path.join(TAB_DIR, "04_rcfs_gamma_sweep.csv"), index=False)

# Aggregator comparison: does the harmonic mean change the ranking?
agg_df = pd.DataFrame(agg_rows).round(4)
agg_df.to_csv(os.path.join(TAB_DIR, "04_aggregator_comparison.csv"), index=False)
rank = (agg_df.assign(rank=agg_df.groupby(["dataset", "aggregator"])["RCFS"]
        .rank(ascending=False, method="min"))
        .pivot_table(index=["dataset", "technique"], columns="aggregator", values="rank"))
print("Technique ranks under each aggregator (1 = best):")
print(rank.to_string())
print(f"EXPL_KSTAR (k*) used throughout: {EXPL_KSTAR}")
print("Notebook 04 complete.")


/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[         German Credit] baseline      AUC=0.781 c_fair=0.885 c_expl=0.348 DI=0.863 |SPD|=0.093 n_eff=28.7 RCFS=0.531


[         German Credit] reweighing    AUC=0.785 c_fair=0.842 c_expl=0.350 DI=0.810 |SPD|=0.126 n_eff=28.6 RCFS=0.532


[         German Credit] reject_option AUC=0.781 c_fair=0.946 c_expl=0.348 DI=0.935 |SPD|=0.043 n_eff=28.7 RCFS=0.536


[         German Credit] exp_gradient  AUC=0.782 c_fair=0.898 c_expl=0.348 DI=0.882 |SPD|=0.086 n_eff=28.7 RCFS=0.533


...

[     Taiwan CC Default] exp_gradient  AUC=0.769 c_fair=0.958 c_expl=0.634 DI=0.950 |SPD|=0.035 n_eff=15.8 RCFS=0.650


[  Credit Card Approval] baseline      AUC=0.692 c_fair=0.992 c_expl=0.456 DI=1.019 |SPD|=0.017 n_eff=21.9 RCFS=0.504


[  Credit Card Approval] reweighing    AUC=0.693 c_fair=0.993 c_expl=0.464 DI=0.992 |SPD|=0.007 n_eff=21.5 RCFS=0.509


[  Credit Card Approval] reject_option AUC=0.692 c_fair=0.986 c_expl=0.456 DI=0.985 |SPD|=0.013 n_eff=21.9 RCFS=0.503


[  Credit Card Approval] exp_gradient  AUC=0.693 c_fair=0.985 c_expl=0.445 DI=0.984 |SPD|=0.013 n_eff=22.5 RCFS=0.499
             dataset     technique    AUC  c_perf  c_fair  c_expl     DI  abs_SPD   n_eff binding_attr  Compliance   RCFS
       German Credit      baseline 0.7813  0.5626  0.8848  0.3483 0.8632   0.0935 28.7104    age_group      0.4999 0.5312
       German Credit    reweighing 0.7849  0.5698  0.8420  0.3502 0.8102   0.1261 28.5542    age_group      0.4947 0.5322
       German Credit reject_option 0.7813  0.5626  0.9458  0.3483 0.9347   0.0430 28.7104    age_group      0.5091 0.5359
       German Credit  exp_gradient 0.7817  0.5633  0.8982  0.3483 0.8821   0.0857 28.7104    age_group      0.5020 0.5326
   Taiwan CC Default      baseline 0.7696  0.5391  0.9544  0.6350 0.9488   0.0399 15.7468    education      0.7627 0.6509
   Taiwan CC Default    reweighing 0.7702  0.5404  0.9566  0.6365 0.9512   0.0381 15.7099    education      0.7644 0.6524
   Taiwan CC Default reject_